# Notebook 2: YOLOv8 Training

This notebook loads the shared dataset split and trains the YOLOv8 model only.

In [1]:
%pip install ultralytics

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import json
import sys
from pathlib import Path

import torch
from ultralytics import YOLO


def resolve_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "medical_detection").exists():
            return candidate

    kaggle_candidate = Path("/kaggle/working/model-comparison-for-lesion-detection")
    if (kaggle_candidate / "medical_detection").exists():
        return kaggle_candidate

    raise FileNotFoundError("Could not locate a project root containing the medical_detection package.")


PROJECT_ROOT = resolve_project_root(Path.cwd())
DATASET_ROOT = Path("/kaggle/input/datasets/capsuleyolo/kyucapsule") if Path("/kaggle/input/datasets/capsuleyolo/kyucapsule").exists() else PROJECT_ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from medical_detection import (
    COMMON_COMPARISON_TRAINING_CONFIG,
    ProjectPaths,
    class_names_from_csv,
    materialize_yolo_dataset,
    write_yolo_data_yaml,
    )

paths = ProjectPaths(project_root=PROJECT_ROOT, dataset_root=DATASET_ROOT)
YOLO_WORKING_DIR = paths.project_root / "yolo_dataset"
YOLO_SPLIT_SUFFIX = "" # _bg15
YOLO_SPLIT_LABEL = YOLO_SPLIT_SUFFIX.lstrip("_") or "baseline"
YOLO_DATA_YAML = paths.project_root / f"data_live_{YOLO_SPLIT_LABEL}.yaml"
YOLO_MODEL_NAME = "yolov8m.pt"
comparison_train_config = COMMON_COMPARISON_TRAINING_CONFIG.copy()

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU detected. Training will run on CPU.")
print(f"PROJECT_ROOT: {paths.project_root}")
print(f"DATASET_ROOT: {paths.dataset_root}")
print(f"YOLO split suffix: {YOLO_SPLIT_SUFFIX or '(baseline)'}")
print("Shared comparison config:", comparison_train_config)

CUDA available: True
GPU: NVIDIA GeForce RTX 5080
PROJECT_ROOT: c:\Users\Domagoj\Downloads\malter
DATASET_ROOT: c:\Users\Domagoj\Downloads\malter
Shared comparison config: {'epochs': 100, 'seed': 42, 'patience': 30, 'optimizer': 'AdamW', 'lr0': 0.001, 'lrf': 0.05, 'weight_decay': 0.0005, 'warmup_epochs': 5, 'cos_lr': True, 'workers': 4}


In [ ]:
materialize_yolo_dataset(
    splits_dir=paths.splits_dir,
    image_dir=paths.image_dir,
    label_dir=paths.label_dir,
    output_dir=YOLO_WORKING_DIR,
    split_suffix=YOLO_SPLIT_SUFFIX,
)

print(f"Materialized split variant: {YOLO_SPLIT_LABEL}")
for split_name in ("train", "val", "test"):
    image_count = len(list((YOLO_WORKING_DIR / "images" / split_name).glob("*")))
    label_count = len(list((YOLO_WORKING_DIR / "labels" / split_name).glob("*.txt")))
    print(f"{split_name}: {image_count} images, {label_count} labels")

train: 12931 images, 12931 labels
val: 3700 images, 3700 labels
test: 1850 images, 1850 labels


In [4]:
class_names = class_names_from_csv(paths.csv_path)
YOLO_DATA_YAML = write_yolo_data_yaml(YOLO_WORKING_DIR, class_names, YOLO_DATA_YAML)
print(f"Created live config at {YOLO_DATA_YAML}")
print(class_names)

Created live config at c:\Users\Domagoj\Downloads\malter\data_live.yaml
['angiodysplasia', 'erosion', 'stenosis', 'lymphangiectasia', 'lymph follicle', 'SMT', 'polyp-like', 'bleeding', 'diverticulum', 'erythema', 'foreign body', 'vein']


In [ ]:
yolo_train_device = list(range(torch.cuda.device_count())) if torch.cuda.device_count() > 1 else (0 if torch.cuda.is_available() else "cpu")
batch_size = 16 if torch.cuda.is_available() else 2
train_imgsz = 640
run_name = f"lesion_yolov8s_{train_imgsz}_{YOLO_SPLIT_LABEL}"

yolo_helper_dir = paths.runs_dir / "detect"
yolo_helper_dir.mkdir(parents=True, exist_ok=True)

model = YOLO(YOLO_MODEL_NAME)
train_args = {
    "data": str(YOLO_DATA_YAML),
    "epochs": comparison_train_config["epochs"],
    "imgsz": train_imgsz,
    "batch": batch_size,
    "device": yolo_train_device,
    "workers": comparison_train_config["workers"] if torch.cuda.is_available() else 0,
    "seed": comparison_train_config["seed"],
    "deterministic": True,
    "patience": comparison_train_config["patience"],
    "optimizer": comparison_train_config["optimizer"],
    "lr0": comparison_train_config["lr0"],
    "lrf": comparison_train_config["lrf"],
    "weight_decay": comparison_train_config["weight_decay"],
    "warmup_epochs": float(comparison_train_config["warmup_epochs"]),
    "cos_lr": comparison_train_config["cos_lr"],
    "project": str(paths.runs_dir / "detect"),
    "name": run_name,
    "hsv_h": 0.01,
    "hsv_s": 0.1,
    "hsv_v": 0.1,
    "multi_scale": False,
}

print(
    f"\nStarting {run_name} with "
    f"imgsz={train_args['imgsz']} batch={train_args['batch']} "
    f"optimizer={train_args['optimizer']} lr0={train_args['lr0']}"
 )
yolo_training_results = model.train(**train_args)
yolo_run_dir = Path(model.trainer.save_dir)
yolo_helper_path = yolo_helper_dir / "latest_yolo_run.txt"
yolo_helper_path.write_text(str(yolo_run_dir), encoding="utf-8")
(yolo_helper_dir / "yolo_training_config.json").write_text(
    json.dumps(train_args, indent=2, default=str),
    encoding="utf-8",
)

print("YOLO latest run helper saved to:")
print(yolo_helper_path)
print("YOLO run directory:")
print(yolo_run_dir)
print("Best weights:")
print(yolo_run_dir / "weights" / "best.pt")


Starting lesion_yolov8s_640 with imgsz=640 batch=16 optimizer=AdamW lr0=0.001
New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.40  Python-3.14.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 5080, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=c:\Users\Domagoj\Downloads\malter\data_live.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.3, hsv_v=0.2, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.05, mask_ratio=

In [6]:
yolo_runs_dir = paths.runs_dir / "detect"
print("YOLO detection runs directory:")
print(yolo_runs_dir)
print("Exists:", yolo_runs_dir.exists())
print("Contents:")
for path in sorted(yolo_runs_dir.glob("*")):
    print(" ", path)

YOLO detection runs directory:
c:\Users\Domagoj\Downloads\malter\runs\detect
Exists: True
Contents:
  c:\Users\Domagoj\Downloads\malter\runs\detect\latest_yolo_run.txt
  c:\Users\Domagoj\Downloads\malter\runs\detect\lesion_yolov8s_640-2
  c:\Users\Domagoj\Downloads\malter\runs\detect\yolo_training_config.json
